# NB_lab_test_trends

Builds `caresyncadb.gold.lab_test_trends` per the STTM (Silver to Gold v1.0). Grain: one row per test type (test_name) per calendar month. Driving table `silver.lab_results` grouped by test_name, year and month, LEFT JOIN to calendar on the first day of the month (`date_key_int`), with a LAG() month-over-month trend. `quarter_name` is derived as 'Q' + calendar.quarter because the silver calendar has no quarter_name column. Load: full rebuild.

In [0]:
%sql
-- Full rebuild: one row per test_name per month (driving table: silver.lab_results)
CREATE OR REPLACE TABLE caresyncadb.gold.lab_test_trends AS
WITH monthly AS (
  SELECT
    test_name,
    YEAR(test_date)                             AS year,
    MONTH(test_date)                            AS month,
    MIN(DATE_TRUNC('MONTH', test_date))         AS month_start,
    COUNT(*)                                    AS total_tests,
    COUNT_IF(result_status = 'Abnormal')        AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')        AS critical_test_count
  FROM caresyncadb.silver.lab_results
  GROUP BY test_name, YEAR(test_date), MONTH(test_date)
),
rates AS (
  SELECT
    m.*,
    CAST(ROUND(abnormal_test_count * 100.0 / total_tests, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(ROUND(critical_test_count * 100.0 / total_tests, 2) AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly m
),
with_lag AS (
  SELECT
    r.*,
    LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct
  FROM rates r
)
SELECT
  w.test_name,
  CAST(w.year AS INT)                                AS year,
  CAST(w.month AS INT)                               AS month,
  c.month_name,
  CONCAT('Q', c.quarter)                             AS quarter_name,
  CAST(w.total_tests AS INT)                         AS total_tests,
  CAST(w.abnormal_test_count AS INT)                 AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)                 AS critical_test_count,
  w.abnormal_rate_pct,
  w.critical_rate_pct,
  CAST(w.prev_month_abnormal_rate_pct AS DECIMAL(5,2)) AS prev_month_abnormal_rate_pct,
  CAST(w.abnormal_rate_pct - w.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct
FROM with_lag w
LEFT JOIN caresyncadb.silver.calendar c
  ON c.date_key_int = DATE_FORMAT(w.month_start, 'yyyyMMdd')

num_affected_rows,num_inserted_rows


In [0]:
# Read the rebuilt gold table and return its record count to the caller (e.g. a job or dbutils.notebook.run)
updated_df = spark.table("caresyncadb.gold.lab_test_trends")
record_count = updated_df.count()
print(f"caresyncadb.gold.lab_test_trends record count: {record_count}")
dbutils.notebook.exit(str(record_count))